##**Mastering the Pinecone vector database.**

##**GOAL: save text sentence embeddings and perform semantic search in Pinecone using Python code.**

Create a Pinecone Account: Head to 'Pinecone.io' and sign up for a free account. After signing up, you’ll be given an API key.

 To start working with Pinecone,
install 'pinecone' library

In [1]:
# place your code here
!pip -q install pinecone sentence-transformers


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


From 'pinecone' library import Pinecone, ServerlessSpec.

In [ ]:
# place your code here
from pinecone import Pinecone, ServerlessSpec

from sentence_transformers import SentenceTransformer
import os
import time

API_KEY = "My_Key"
pc = Pinecone(api_key=API_KEY)

model = SentenceTransformer("paraphrase-MiniLM-L6-v2")
dimension = model.get_sentence_embedding_dimension()

index_name = "day9-vector-search"

existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created")
else:
    print("Index already exists")

while not pc.describe_index(index_name).status["ready"]:
    time.sleep(1)

index = pc.Index(index_name)

c:\Users\mithsuka.dikkumbura\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Index created


On the Pinecone main page, navigate to the "Get started" section via the left menu. Next, follow the "Quick Start for Developers" guide.
* Initialaze a client.
* Create an index.
* Upsert data.
* Perform a search.









When you perform "Upsert data", add a list of your own text strings.

When you perform "Search", use of your own text string.

In [8]:
# place your code here
texts = [
    "Route optimization improves delivery efficiency by selecting the shortest and fastest path.",
    "Predictive analytics helps businesses forecast demand using historical sales data.",
    "Semantic search compares meaning between texts using vector embeddings.",
    "Cybersecurity monitoring systems detect unusual network traffic patterns.",
    "Good user interface design reduces user frustration and improves usability.",
    "Data engineering pipelines ensure accurate and reliable data processing."
]

vectors = model.encode(texts).tolist()

data_to_upsert = [
    {
        "id": f"text-{i}",
        "values": vectors[i],
        "metadata": {"text": texts[i]}
    }
    for i in range(len(texts))
]

index.upsert(vectors=data_to_upsert)

print(f"Successfully upserted {len(data_to_upsert)} ")

query_text = "How can machine learning improve delivery performance?"

query_vector = model.encode(query_text).tolist()

results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

Successfully upserted 6 


Print search results.

In [9]:
# place your code here
print("Query:", query_text)
print("\nTop 3 Most Similar Results:\n")

for rank, match in enumerate(results["matches"], start=1):
    print(f"{rank}) Score: {match['score']:.4f}")
    print("   Text :", match["metadata"]["text"])
    print("-" * 60)

Query: How can machine learning improve delivery performance?

Top 3 Most Similar Results:

1) Score: 0.5162
   Text : Route optimization improves delivery efficiency by selecting the shortest and fastest path.
------------------------------------------------------------
2) Score: 0.3465
   Text : Predictive analytics helps businesses forecast demand using historical sales data.
------------------------------------------------------------
3) Score: 0.3164
   Text : Data engineering pipelines ensure accurate and reliable data processing.
------------------------------------------------------------
